In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pandas.core.algorithms")

In [ ]:
import os
from pathlib import Path
import pandas as pd
import logging
import numpy as np
import torch
import scanpy as sc

from cellwhisperer.config import get_path
from cellwhisperer.utils.inference import score_transcriptomes_vs_texts
from cellwhisperer.utils.model_io import load_cellwhisperer_model

# Configure logging
logging.basicConfig(level=logging.INFO)

In [ ]:
#### Parameters ####
ckpt_file_path = snakemake.input.model
metadata_col = snakemake.wildcards.metadata_col
dataset_name = snakemake.wildcards.dataset
grouping = snakemake.wildcards.grouping
average_by_class = snakemake.params.average_by_class

print(f"Processing dataset: {dataset_name}")
print(f"Metadata column: {metadata_col}")
print(f"Grouping: {grouping}")
print(f"Average by class: {average_by_class}")

In [ ]:
#### Load model ####
(
    pl_model_cellwhisperer,
    text_processor_cellwhisperer,
    cellwhisperer_transcriptome_processor,
    cellwhisperer_image_processor
) = load_cellwhisperer_model(model_path=ckpt_file_path, eval=True)
cellwhisperer_model = pl_model_cellwhisperer.model

print("Model loaded successfully")

In [ ]:
#### Load spatial transcriptomics data ####
adata = sc.read_h5ad(snakemake.input.raw_read_count_table)
print(f"Loaded dataset with shape: {adata.shape}")
print(f"Available obs columns: {list(adata.obs.columns)}")

print(f"Metadata column '{metadata_col}' unique values: {adata.obs[metadata_col].value_counts()}")

In [ ]:
#### Load processed embeddings
processed_data = np.load(snakemake.input.processed_dataset)
print(f"Loaded processed data with keys: {list(processed_data.keys())}")

# Add embeddings to adata
if 'transcriptome_embeds' in processed_data:
    adata.obsm['X_transcriptome_embeddings'] = processed_data['transcriptome_embeds']
    print(f"Added transcriptome embeddings with shape: {adata.obsm['X_transcriptome_embeddings'].shape}")

# Add image embeddings 
assert 'image_embeds' in processed_data
adata.obsm['X_image_embeddings'] = processed_data['image_embeds']
print(f"Added image embeddings with shape: {adata.obsm['X_image_embeddings'].shape}")


In [ ]:
#### Prepare text queries for prediction ####
# Get unique classes from the metadata column
unique_classes = adata.obs[metadata_col].unique()
unique_classes = [c for c in unique_classes if pd.notna(c) and c not in snakemake.params["filter_classes"]] # ["UNASSIGNED"]] # 
print(f"Found {len(unique_classes)} unique classes: {unique_classes}")

# Create text descriptions for each class
if metadata_col == 'cell_type_annotations':
    text_list = [f"Sample of a {class_name} cell" for class_name in unique_classes]  # TODO
elif metadata_col == 'region_type_expert_annotation':
    # Map abbreviations to full names for better text representation
    region_mapping = {
        'NOR': 'normal cells',
        'TUM': 'tumor cells', 
        'TLS': 'tertiary lymphoid structures',
        'INFL': 'infiltrating cells'
    }
    text_list = [f"{region_mapping.get(class_name, class_name)}" for class_name in unique_classes]
else:
    text_list = [f"Sample with {metadata_col} of {class_name}" for class_name in unique_classes]

print(f"Text queries: {text_list}")

In [ ]:
#### Make predictions ####
# Filter data to only include cells with valid annotations
valid_mask = adata.obs[metadata_col].isin(unique_classes)
adata_filtered = adata[valid_mask].copy()
print(f"Filtered to {adata_filtered.shape[0]} cells with valid annotations")

# Perform zero-shot prediction
scores, true_classes = score_transcriptomes_vs_texts(
    transcriptome_input=torch.from_numpy(adata_filtered.obsm['X_image_embeddings']),
    text_list_or_text_embeds=text_list,
    model=cellwhisperer_model,
    batch_size=32,
    logit_scale=cellwhisperer_model.discriminator.temperature.exp(),
    average_mode="embeddings" if average_by_class else None,
    grouping_keys=adata.obs[metadata_col].values,  # only relevant if average_mode is not None, but needed for return value true_classes
    score_norm_method=None
)    

In [ ]:
text_list

In [ ]:
pd.Series(scores.argmax(axis=0)).value_counts()

In [ ]:

# scores_T = (scores/scores.norm(dim=0)).T  # n_cells * n_text
predicted_labels = [unique_classes[x] for x in scores.T.argmax(axis=1)]

# %%
result_df = pd.DataFrame(index=true_classes if snakemake.params.average_by_class else adata_filtered.obs.index)
for term in text_list:
    if snakemake.params.average_by_class:
        result_df[f"score_for_{term}"] = scores.T[:, text_list.index(term)].tolist()
    else:
        result_df[f"score_for_{term}"] = adata_filtered.obs[metadata_col].apply(dict(zip(true_classes, scores_T[:, text_list.index(term)].numpy())).get)

if snakemake.params.average_by_class:
    raise NotImplementedError("not sure if the below makes sense")
    result_df["predicted_labels"] = adata_filtered.obs[metadata_col].apply(dict(zip(true_classes, predicted_labels)).get)
    result_df["label"] = adata_filtered.obs[metadata_col]
else:
    result_df["predicted_labels"] = predicted_labels
    result_df["label"] = adata_filtered.obs[metadata_col]

result_df["is_correct"] = (result_df["predicted_labels"] == result_df["label"])

# Store the results
result_df.to_csv(snakemake.output.predictions, index=True)


In [ ]:
# Print summary
accuracy = (result_df['label'] == result_df['predicted_labels']).mean()
print(f"Final accuracy: {accuracy:.3f}")

# Print confusion matrix summary
confusion_summary = pd.crosstab(result_df['label'], result_df['predicted_labels'])
print("\nConfusion matrix:")
print(confusion_summary)

In [ ]:

# Add scores for each class to adata.obs
for i, class_name in enumerate(unique_classes):
    adata_filtered.obs[f'score_{class_name}'] = scores[i].numpy()



In [ ]:
# Plot UMAP colored by each class score
import matplotlib.pyplot as plt
for class_name in unique_classes:
    sc.pl.embedding(adata_filtered, color=f'score_{class_name}', cmap='viridis', title=f"Score: {class_name}", use_raw=False, show=False, basis="spatial", dimensions=[(1,0)])
    plt.show()

In [ ]:
adata_filtered